# Paradata Feature Extraction Pipeline - Example Analysis

This notebook demonstrates how to use the feature extraction pipeline on mobile survey paradata.

In [ ]:
import pandas as pd
import numpy as np
import os
import sys

# Add the parent directory to the path so we can import our modules
sys.path.append(os.path.abspath('..'))

from src.parsers.surveycto_parser import parse_surveycto_audit
from src.features.response_time import calculate_basic_timing, calculate_speed_index
from src.features.pauses import calculate_pauses
from src.features.navigation import calculate_navigation_metrics
from src.features.rhythm import calculate_rhythm_metrics
from src.validation.statistical_tests import compare_groups
from src.visualization.timing_plots import plot_feature_distribution, plot_correlation_matrix

## 1. Data Parsing
Here we simulate parsing a SurveyCTO audit log.

In [ ]:
# In a real scenario, you would point this to your actual audit.csv file
# parsed_df = parse_surveycto_audit('../data/audit.csv')
# display(parsed_df.head())

# Create mock parsed data for demonstration
times = pd.date_range('2023-01-01 10:00:00', periods=10, freq='10s')
mock_df = pd.DataFrame({
    'event_type': ['Question'] * 10,
    'question_name': [f'q{i}' for i in range(1, 11)],
    'start_time': times,
    'end_time': times + pd.Timedelta(seconds=5),
    'duration_ms': [5000, 3000, 15000, 1000, 4500, 6000, 2000, 9000, 3500, 4000]
})
display(mock_df.head())

## 2. Feature Extraction
Calculate timing features for this mock interview.

In [ ]:
basic_timing = calculate_basic_timing(mock_df)
print("Basic Timing:", basic_timing)

speed_idx = calculate_speed_index(basic_timing['total_duration_ms'], 50000.0) # Assume global median is 50000
print(f"Speed Index: {speed_idx:.2f}")

pauses = calculate_pauses(mock_df, pause_threshold_ms=4000)
print("Pauses:", pauses)

nav = calculate_navigation_metrics(mock_df, straightline_threshold_ms=2000)
print("Navigation:", nav)

rhythm = calculate_rhythm_metrics(mock_df, window_size=3)
print("Rhythm:", rhythm)

## 3. Visualization & Validation
Generating arbitrary feature data across multiple interviews to test visualization and validation.

In [ ]:
np.random.seed(42)
n_interviews = 100

# 0 = high quality, 1 = problematic
interviews_df = pd.DataFrame({
    'interview_id': range(n_interviews),
    'total_duration_ms': np.random.normal(600000, 100000, n_interviews),
    'quality_flag': np.random.binomial(1, 0.2, n_interviews)
})

# Problematic interviews might be faster (speeding)
interviews_df.loc[interviews_df['quality_flag'] == 1, 'total_duration_ms'] *= 0.6

# Run statistical test
test_res = compare_groups(interviews_df, 'total_duration_ms', 'quality_flag')
print("T-Test Results:", test_res)

# Plot distribution
fig = plot_feature_distribution(interviews_df, 'total_duration_ms', 'quality_flag', title='Total Duration by Quality')
display(fig)